# 10.4: Bidirectional Recurrent Neural Networks

In [2]:
import torch
from torch import nn
from d2l import torch as d2l

## 10.4.1: Implementation from Scratch

In [3]:
class BiRNNScratch(d2l.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        # Reminder: sigma is used for weight initialization
        self.f_rnn = d2l.RNNScratch(num_inputs, num_hiddens, sigma)
        self.b_rnn = d2l.RNNScratch(num_inputs, num_hiddens, sigma)
        self.num_hiddens *= 2 # Output dimension will be doubled

In [4]:
@d2l.add_to_class(BiRNNScratch)
def forward(self, inputs, Hs=None):
    f_H, b_H = Hs if Hs is not None else (None, None)
    f_outputs, f_H = self.rnn(inputs, f_H)
    b_outputs, b_H = self.rnn(reversed(inputs), b_H)
    outputs = [torch.cat((f,b), -1) for f,b in zip(
        f_outputs, reversed(b_outputs))] # Concatenate results along last dimension (h, n x h)
    return outputs, (f_H, b_H)

## 10.4.2: Concise Implementation

In [5]:
class BiGRU(d2l.RNN):
    def __init__(self, num_inputs, num_hiddens):
        d2l.Module.__init__(self)
        self.save_hyperparameters()
        self.rnn = nn.GRU(num_inputs, num_hiddens, bidirectional=True)
        self.num_hiddens *= 2

# Summary

- In bidirectional RNNs, hidden state for each time step is simultaneously determined by data prior and after the current time step
- Bidirectional RNNs are mostly useful for sequence encoding and estimation of observations given bidirectional context
    - e.g. masking 
- Bidirectional RNNs are very costly to train due to long gradient chains